# Fase 2 — Cálculo de métricas

Lee los archivos `.pkl` de cada corrida y calcula **HV**, **GD** e **IGD**.

- **Frente de referencia:** por categoría (variante × seed_mode × N) — unión de las 30 corridas del grupo, no-dominados.
- **Punto de referencia HV:** `[0, 0]` (consistente con el TFG).
- **Salida:** `results/metrics.csv`

In [1]:
import os
import glob
import pickle
import re
import numpy as np
import pandas as pd

from pymoo.indicators.hv import Hypervolume
from pymoo.indicators.gd import GD
from pymoo.indicators.igd import IGD
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

RESULTS_BASE = './results'
N_VALUES     = [50, 100, 150, 200]
VARIANTES    = ['original', 'modified']
SEED_MODES   = ['same_seed', 'diff_seed']

## 1. Cargar todos los archivos `.pkl`

In [2]:
def load_all_runs():
    runs = []
    for variante in VARIANTES:
        for seed_mode in SEED_MODES:
            for n in N_VALUES:
                folder = os.path.join(RESULTS_BASE, variante, seed_mode, f'N{n}')
                if not os.path.exists(folder):
                    continue
                pkls = sorted(glob.glob(os.path.join(folder, '*.pkl')))
                for i, pkl_path in enumerate(pkls):
                    with open(pkl_path, 'rb') as f:
                        data = pickle.load(f)
                    m = re.search(r'seed_(\d+)_', os.path.basename(pkl_path))
                    seed = int(m.group(1)) if m else None
                    runs.append({
                        'variante':  variante,
                        'seed_mode': seed_mode,
                        'N':         n,
                        'corrida':   i + 1,
                        'seed':      seed,
                        'fitnessF':  np.array(data['fitnessF']),
                    })
    return runs

runs = load_all_runs()
print(f'Total de corridas cargadas: {len(runs)}')

summary = pd.DataFrame([
    {'variante': r['variante'], 'seed_mode': r['seed_mode'], 'N': r['N']}
    for r in runs
]).groupby(['variante', 'seed_mode', 'N']).size().rename('corridas')
print(summary.to_string())

Total de corridas cargadas: 480
variante  seed_mode  N  
modified  diff_seed  50     30
                     100    30
                     150    30
                     200    30
          same_seed  50     30
                     100    30
                     150    30
                     200    30
original  diff_seed  50     30
                     100    30
                     150    30
                     200    30
          same_seed  50     30
                     100    30
                     150    30
                     200    30


## 2. Construir frentes de referencia por categoría (variante × seed_mode × N)

In [3]:
from collections import defaultdict

nds = NonDominatedSorting()

# Agrupar corridas por (variante, seed_mode, N)
groups = defaultdict(list)
for r in runs:
    key = (r['variante'], r['seed_mode'], r['N'])
    groups[key].append(r['fitnessF'])

# Para cada grupo: unión de las 30 corridas → no-dominados → frente de referencia
ref_fronts = {}
for key, fronts in groups.items():
    union_F   = np.vstack(fronts)
    front_idx = nds.do(union_F, only_non_dominated_front=True)
    ref_fronts[key] = union_F[front_idx]

for (variante, seed_mode, n), rf in sorted(ref_fronts.items()):
    print(f'{variante:10s}  {seed_mode:10s}  N={n:3d}  →  {len(rf):4d} puntos en el frente de referencia')

modified    diff_seed   N= 50  →   601 puntos en el frente de referencia
modified    diff_seed   N=100  →   999 puntos en el frente de referencia
modified    diff_seed   N=150  →  1209 puntos en el frente de referencia
modified    diff_seed   N=200  →  1482 puntos en el frente de referencia
modified    same_seed   N= 50  →   602 puntos en el frente de referencia
modified    same_seed   N=100  →   986 puntos en el frente de referencia
modified    same_seed   N=150  →  1241 puntos en el frente de referencia
modified    same_seed   N=200  →  1393 puntos en el frente de referencia
original    diff_seed   N= 50  →   646 puntos en el frente de referencia
original    diff_seed   N=100  →  1140 puntos en el frente de referencia
original    diff_seed   N=150  →  1397 puntos en el frente de referencia
original    diff_seed   N=200  →  1671 puntos en el frente de referencia
original    same_seed   N= 50  →   688 puntos en el frente de referencia
original    same_seed   N=100  →  1128 puntos en el

## 3. Calcular HV, GD e IGD por corrida

In [4]:
hv_indicator = Hypervolume(ref_point=np.zeros(2))

records = []
for r in runs:
    F   = r['fitnessF']
    key = (r['variante'], r['seed_mode'], r['N'])
    rf  = ref_fronts[key]

    records.append({
        'variante':  r['variante'],
        'seed_mode': r['seed_mode'],
        'N':         r['N'],
        'corrida':   r['corrida'],
        'seed':      r['seed'],
        'HV':        hv_indicator.do(F),
        'GD':        GD(rf).do(F),
        'IGD':       IGD(rf).do(F),
    })

metrics_df = pd.DataFrame(records)
out_path = os.path.join(RESULTS_BASE, 'metrics.csv')
metrics_df.to_csv(out_path, index=False)
print(f'Guardado: {os.path.abspath(out_path)}')
metrics_df.head()

Guardado: c:\Projects\tfg\src\results\metrics.csv


,variante,seed_mode,N,corrida,seed,HV,GD,IGD
0,original,same_seed,50,1,10,0.566413,0.000193,0.009292
1,original,same_seed,50,2,11,0.564370,0.001945,0.009095
2,original,same_seed,50,3,12,0.563156,0.001923,0.009945
3,original,same_seed,50,4,13,0.563248,0.003013,0.009407
4,original,same_seed,50,5,14,0.563261,0.002140,0.009756


## 4. Resumen estadístico por grupo

In [5]:
metrics_df.groupby(['variante', 'seed_mode', 'N'])[['HV', 'GD', 'IGD']].agg(['mean', 'std', 'median'])

HV                            GD            \
                            mean       std    median      mean       std   
variante seed_mode N                                                       
modified diff_seed 50   0.582795  0.002314  0.583303  0.002124  0.001409   
                   100  0.592600  0.000974  0.592948  0.001302  0.000669   
                   150  0.596339  0.000457  0.596292  0.001160  0.000278   
                   200  0.598367  0.000386  0.598422  0.001043  0.000258   
         same_seed 50   0.583216  0.003326  0.583601  0.002274  0.002165   
                   100  0.592920  0.000824  0.593130  0.001285  0.000563   
                   150  0.596346  0.000534  0.596388  0.001148  0.000352   
                   200  0.598114  0.000823  0.598249  0.001213  0.000547   
original diff_seed 50   0.563553  0.004219  0.564400  0.002179  0.002949   
                   100  0.573164  0.000786  0.573294  0.000900  0.000445   
                   150  0.576074  0.000462  0.576141  0.000802  0.000296   
                   200  0.577716  0.000401  0.577819  0.000764  0.000266   
         same_seed 50   0.564358  0.003999  0.565068  0.001838  0.002801   
                   100  0.573404  0.000650  0.573517  0.000912  0.000425   
                   150  0.576119  0.000622  0.576262  0.000894  0.000443   
                   200  0.577765  0.000446  0.577860  0.000709  0.000272   

                                       IGD                      
                          median      mean       std    median  
variante seed_mode N                                            
modified diff_seed 50   0.001873  0.009305  0.000993  0.008986  
                   100  0.001116  0.004715  0.000427  0.004705  
                   150  0.001220  0.003256  0.000177  0.003264  
                   200  0.001036  0.002498  0.000159  0.002447  
         same_seed 50   0.001909  0.009371  0.001300  0.009057  
                   100  0.001119  0.004693  0.000352  0.004607  
                   150  0.001086  0.003247  0.000205  0.003203  
                   200  0.001086  0.002599  0.000372  0.002481  
original diff_seed 50   0.001448  0.009687  0.001912  0.009335  
                   100  0.000785  0.004675  0.000365  0.004683  
                   150  0.000794  0.003156  0.000154  0.003132  
                   200  0.000751  0.002411  0.000163  0.002388  
         same_seed 50   0.001032  0.009567  0.001802  0.009345  
                   100  0.000816  0.004560  0.000282  0.004492  
                   150  0.000806  0.003162  0.000242  0.003143  
                   200  0.000664  0.002398  0.000176  0.002351